<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 필요 패키지 설치
!pip install pyreadstat

# 2. 구글 드라이브 연결
from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat

drive.mount("/content/drive")

# 3. 데이터 로드 (W, STRATA, CLUSTER 포함)
DATA_DIR = "/content/drive/MyDrive/hdac_data"
cols = [
    "YEAR",
    "STRATA",
    "CLUSTER",
    "W",
    "GRADE",
    "SEX",
    "AGE",
    "HT",
    "WT",
    "F_BR",
]
df, meta = pyreadstat.read_sas7bdat(
    f"{DATA_DIR}/kyrbs2025.sas7bdat", usecols=cols
)

print("=== 1. 복합표본 설계변수 점검 (원자료 전체 $N=54,170$) ===")
print(f"• 전체 응답자 수: {len(df):,}명")
print(f"• 가중치(W) 결측 수: {df['W'].isna().sum()}개 (유효값 {df['W'].notna().sum():,}개)")
print(
    f"• 층(STRATA) 수: {df['STRATA'].nunique()}개 (결측 {df['STRATA'].isna().sum()}개)"
)
print(
    f"• 집락(CLUSTER) 수: {df['CLUSTER'].nunique()}개 (결측 {df['CLUSTER'].isna().sum()}개)"
)

# 4. 도메인 변수 정의 (고등학생 = 1, 중학생 = 0)
df["domain_hs"] = np.where(df["GRADE"].isin([4, 5, 6]), 1, 0)

print("\n=== 2. 도메인 분석 대상 인원 (GRADE 기준) ===")
print(
    df["domain_hs"]
    .value_counts()
    .rename({1: "고등학생 도메인(분석대상)", 0: "중학생(도메인외)"})
)

# 5. 고등학생 도메인 내 노출 및 결과변수 결측/유효 인원 계산
df_hs = df[df["domain_hs"] == 1].copy()

# (1) 노출변수 F_BR 점검
f_br_missing = df_hs["F_BR"].isna().sum() + (df_hs["F_BR"] == 9).sum()

# (2) 결과변수 HT, WT 점검
ht_wt_missing = (
    df_hs["HT"].isna()
    | df_hs["WT"].isna()
    | (df_hs["HT"] == 999)
    | (df_hs["WT"] == 999)
).sum()

print("\n=== 3. 고등학생 도메인 내 단계별 제외 인원 및 유효 인원 ===")
print(f"• 고등학생 총 도메인 인원: {len(df_hs):,}명")
print(f"• 노출변수(F_BR) 결측/무응답 제외: {f_br_missing}명")
print(f"• 결과변수(HT/WT) 결측/이상치 제외: {ht_wt_missing}명")
print(f"• 최종 노출-결과 유효 분석 인원: {len(df_hs) - ht_wt_missing:,}명")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 8.6 MB/s  0:00:00
Mounted at /content/drive
=== 1. 복합표본 설계변수 점검 (원자료 전체 $N=54,170$) ===
• 전체 응답자 수: 54,170명
• 가중치(W) 결측 수: 0개 (유효값 54,170개)
• 층(STRATA) 수: 106개 (결측 0개)
• 집락(CLUSTER) 수: 796개 (결측 0개)

=== 2. 도메인 분석 대상 인원 (GRADE 기준) ===
domain_hs
중학생(도메인외)         28327
고등학생 도메인(분석대상)    25843
Name: count, dtype: int64

=== 3. 고등학생 도메인 내 단계별 제외 인원 및 유효 인원 ===
• 고등학생 총 도메인 인원: 25,843명
• 노출변수(F_BR) 결측/무응답 제외: 0명
• 결과변수(HT/WT) 결측/이상치 제외: 766명
• 최종 노출-결과 유효 분석 인원: 25,077명
